# Ultra-short standalone pseudobulk DE pipeline

No project imports: this notebook directly aggregates pseudobulk counts, runs PyDESeq2, and computes optional Reactome/GMT ORA and GSEA outputs.


In [46]:
from pathlib import Path

H5AD_PATH = "/Users/bastien.herve/Downloads/RRMAP2_xenium_all_samples.cellcharter.companion.ready.with_metadata.rerun.with_AnnoL1Curated_with_Region_Anno2to4Updated.h5ad"
GROUPBY = "Anno_L1_curated"
REPLICATE = "meta_sample_id"
SOURCE = "OPC"
REFERENCE = "Schwann cell"

COUNTS_LAYER = "counts"
MIN_CELLS = 20
MIN_CELL_COUNTS = 100
MIN_GENE_COUNTS = 100
MIN_REPLICATES = 3
MIN_PCT_EXPRESSED = 0.1
P_ADJUST_METHOD = "fdr_bh"
PADJ_CUTOFF = 0.05
LOG2FC_CUTOFF = 2
DESEQ2_FIT_TYPE = "mean"
N_CPUS = 4

PATHWAY_GMT = None
PATHWAY_ORGANISM = "Mouse"
PATHWAY_TOP_N = 20
PATHWAY_MIN_OVERLAP = 3
PATHWAY_MAX_SIZE = 500
PATHWAY_GSEA_PERMUTATIONS = 100
PATHWAY_GSEA_SEED = 0

stem = Path(H5AD_PATH).with_suffix("")
OUT_PREFIX = stem.parent / f"{stem.name}.{GROUPBY}.{SOURCE}_vs_{REFERENCE}"

In [28]:
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests


def dense_integer_counts(x):
    x = x.toarray() if sp.issparse(x) else np.asarray(x)
    x = np.nan_to_num(np.asarray(x, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    return np.rint(np.clip(x, 0, None)).astype(int)


def row_sum(x):
    return (
        np.asarray(x.sum(axis=1)).ravel()
        if sp.issparse(x)
        else np.asarray(x).sum(axis=1)
    )


def padjust(pvalues, method):
    pvalues = np.asarray(pvalues, dtype=float)
    out = np.full(pvalues.shape, np.nan, dtype=float)
    ok = np.isfinite(pvalues)
    method = str(method or "fdr_bh").replace("-", "_").lower()
    out[ok] = (
        pvalues[ok]
        if method in {"none", "raw"}
        else multipletests(np.clip(pvalues[ok], 0, 1), method=method)[1]
    )
    return out


def pct_positive(x, mask, cols):
    sub = x[np.asarray(mask, dtype=bool)][:, list(cols)]
    return (
        np.asarray((sub > 0).mean(axis=0)).ravel()
        if sp.issparse(sub)
        else np.mean(np.asarray(sub) > 0, axis=0)
    )


def clean_gene_set(genes):
    seen, cleaned = set(), []
    for gene in genes:
        gene = str(gene).strip()
        key = gene.lower()
        if gene and key not in seen:
            seen.add(key)
            cleaned.append(gene)
    return cleaned


def read_gmt(paths):
    paths = [paths] if isinstance(paths, (str, Path)) else list(paths)
    gene_sets = {}
    for path in paths:
        with Path(path).expanduser().open(encoding="utf-8", errors="replace") as handle:
            for line in handle:
                term, _, *genes = line.rstrip("\n\r").split("\t")
                if term and genes:
                    gene_sets[term] = clean_gene_set(genes)
    return gene_sets


def ora(genes, gene_sets, universe):
    selected = {str(g).lower() for g in genes}
    universe = {str(g).lower() for g in universe}
    rows = []
    for term, members in gene_sets.items():
        pathway = {str(g).lower() for g in members} & universe
        overlap = pathway & selected
        if len(overlap) >= PATHWAY_MIN_OVERLAP and len(pathway) <= PATHWAY_MAX_SIZE:
            pvalue = hypergeom.sf(
                len(overlap) - 1, len(universe), len(pathway), len(selected)
            )
            rows.append(
                {
                    "Term": term,
                    "Count": len(overlap),
                    "GeneRatio": len(overlap) / max(len(selected), 1),
                    "P-value": pvalue,
                }
            )
    table = pd.DataFrame(rows)
    if len(table):
        table["Adjusted P-value"] = padjust(table["P-value"], "fdr_bh")
        table["-log10 adjusted P-value"] = -np.log10(
            np.clip(table["Adjusted P-value"], np.nextafter(0, 1), 1)
        )
        table = table.sort_values(
            ["GeneRatio", "Adjusted P-value"], ascending=[False, True]
        ).head(PATHWAY_TOP_N)
    return table

In [29]:
adata = ad.read_h5ad(H5AD_PATH)
counts = adata.layers[COUNTS_LAYER] if COUNTS_LAYER else adata.X
group = adata.obs[GROUPBY].astype("category")
replicate = adata.obs[REPLICATE].astype(str)

In [30]:
valid = (group.cat.codes.to_numpy() >= 0) & replicate.notna().to_numpy()
valid &= row_sum(counts) >= int(MIN_CELL_COUNTS)
valid_idx = np.flatnonzero(valid)
valid_cells = pd.DataFrame(
    {
        "replicate": replicate.to_numpy()[valid],
        "group": group.astype(str).to_numpy()[valid],
    }
)
sample_key = valid_cells["replicate"] + "\x1f" + valid_cells["group"]
sample_codes, _ = pd.factorize(sample_key, sort=False)

In [31]:
incidence = sp.csr_matrix(
    (np.ones(len(valid_idx)), (sample_codes, valid_idx)),
    shape=(sample_codes.max() + 1, adata.n_obs),
)
pb_counts = dense_integer_counts(incidence @ counts)
pb_meta = valid_cells.groupby(sample_key, sort=False).agg(
    _pb_replicate=("replicate", "first"),
    _pb_group=("group", "first"),
    n_cells=("group", "size"),
)
pb_meta.index = [f"pb_{i}" for i in range(len(pb_meta))]

In [32]:
categories = [str(category) for category in group.cat.categories]
retained = [
    category
    for category in categories
    if pb_meta.loc[
        (pb_meta["_pb_group"] == category) & (pb_meta["n_cells"] >= MIN_CELLS),
        "_pb_replicate",
    ].nunique()
    >= MIN_REPLICATES
]
model_mask = pb_meta["_pb_group"].isin(retained) & (pb_meta["n_cells"] >= MIN_CELLS)
model_counts = pb_counts[model_mask.to_numpy()]
model_meta = pb_meta.loc[model_mask, ["_pb_replicate", "_pb_group"]].copy()
model_meta["_pb_replicate"] = pd.Categorical(model_meta["_pb_replicate"].astype(str))
model_meta["_pb_group"] = pd.Categorical(
    model_meta["_pb_group"].astype(str), categories=retained
)

In [33]:
gene_keep = model_counts.sum(axis=0) >= int(MIN_GENE_COUNTS)
model_counts = model_counts[:, gene_keep]
model_genes = adata.var_names.astype(str)[gene_keep]
model_pairs = set(
    zip(model_meta["_pb_replicate"].astype(str), model_meta["_pb_group"].astype(str))
)
model_cell_mask = valid & np.fromiter(
    (
        (str(r), str(g)) in model_pairs
        for r, g in zip(replicate.to_numpy(), group.astype(str).to_numpy())
    ),
    dtype=bool,
    count=adata.n_obs,
)
source_cell_mask = model_cell_mask & (group.astype(str).to_numpy() == str(SOURCE))
reference_cell_mask = model_cell_mask & (group.astype(str).to_numpy() == str(REFERENCE))

print(
    f"Pseudobulk model: {model_counts.shape[0]:,} samples x {model_counts.shape[1]:,} genes; retained categories: {len(retained):,}"
)

Pseudobulk model: 2,042 samples x 5,101 genes; retained categories: 16


In [ ]:
dds = DeseqDataSet(
    counts=pd.DataFrame(model_counts, index=model_meta.index, columns=model_genes),
    metadata=model_meta,
    design="~ _pb_replicate + _pb_group",
    fit_type=DESEQ2_FIT_TYPE,
    n_cpus=N_CPUS,
    quiet=True,
)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    try:
        dds.deseq2(fit_type=DESEQ2_FIT_TYPE)
    except TypeError:
        dds.deseq2()

if "LFC" not in dds.varm:
    raise RuntimeError('DESeq2 fitting did not complete: dds.varm["LFC"] is missing.')

/Users/bastien.herve/miniconda3/envs/karospace-pseudobulk/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [ ]:
if "LFC" not in dds.varm:
    raise RuntimeError(
        "dds is not fitted; rerun the previous cell containing dds.deseq2()."
    )

contrast = np.asarray(
    dds.contrast(column="_pb_group", baseline=REFERENCE, group_to_compare=SOURCE),
    dtype=float,
)
stats = DeseqStats(dds, contrast=contrast, quiet=True, n_cpus=N_CPUS)
stats.summary()

In [47]:
raw = stats.results_df.copy()
raw["padj"] = padjust(raw["pvalue"], P_ADJUST_METHOD)

gene_pos = {str(gene): i for i, gene in enumerate(adata.var_names.astype(str))}
result_genes = raw.index.astype(str).to_list()
result_cols = [gene_pos[gene] for gene in result_genes]
min_pct = (
    float(MIN_PCT_EXPRESSED) / 100
    if float(MIN_PCT_EXPRESSED) > 1
    else float(MIN_PCT_EXPRESSED)
)

de_table = pd.DataFrame(
    {
        "gene": result_genes,
        "baseMean": raw["baseMean"].to_numpy(float),
        "log2FoldChange": raw["log2FoldChange"].to_numpy(float),
        "stat": raw["stat"].to_numpy(float),
        "pvalue": raw["pvalue"].to_numpy(float),
        "padj": raw["padj"].to_numpy(float),
        "pct_source": pct_positive(counts, source_cell_mask, result_cols),
        "pct_reference": pct_positive(counts, reference_cell_mask, result_cols),
    }
)
de_table["max_pct"] = de_table[["pct_source", "pct_reference"]].max(axis=1)
de_table["is_de"] = (
    (de_table["padj"] <= PADJ_CUTOFF)
    & (de_table["log2FoldChange"].abs() >= LOG2FC_CUTOFF)
    & (de_table["max_pct"] >= min_pct)
)
de_table = de_table.sort_values(
    ["padj", "pvalue", "log2FoldChange"], ascending=[True, True, False]
).reset_index(drop=True)
de_table.head(20)

,gene,baseMean,log2FoldChange,stat,pvalue,padj,pct_source,pct_reference,max_pct,is_de
0,Gpr17,143.636317,8.468806,78.085843,0.0,0.0,0.808975,0.007243,0.808975,True
1,Tnr,247.346984,8.217038,94.838507,0.0,0.0,0.960014,0.011689,0.960014,True
2,Myt1,65.990233,7.814341,70.042828,0.0,0.0,0.677700,0.004641,0.677700,True
3,Vxn,39.130940,7.539832,56.433077,0.0,0.0,0.535566,0.002505,0.535566,True
4,Olig1,107.183986,7.081222,78.319162,0.0,0.0,0.762914,0.007456,0.762914,True
5,Olig2,110.489646,6.990979,81.384873,0.0,0.0,0.829506,0.010039,0.829506,True
6,Cspg5,189.514898,6.960266,109.365943,0.0,0.0,0.736198,0.018427,0.736198,True
7,Dscam,40.431121,6.815946,59.310966,0.0,0.0,0.499156,0.003612,0.499156,True
8,C1ql1,99.395349,6.778036,75.453292,0.0,0.0,0.681231,0.010446,0.681231,True
9,Megf11,16.900076,6.773766,41.788265,0.0,0.0,0.317146,0.001883,0.317146,True


In [48]:
de_table["is_de"].value_counts()

is_de
False    4643
True      458
Name: count, dtype: int64

In [49]:
import gseapy as gp

if PATHWAY_GMT:
    gene_sets = read_gmt(PATHWAY_GMT)
    pathway_source = {"source": "gmt", "gene_set_count": len(gene_sets)}
else:
    libraries = gp.get_library_name(organism=PATHWAY_ORGANISM)
    reactome_libraries = [name for name in libraries if "reactome" in str(name).lower()]
    reactome_library = sorted(
        reactome_libraries,
        key=lambda name: (
            max(
                [
                    int(x)
                    for x in "".join(
                        ch if ch.isdigit() else " " for ch in str(name)
                    ).split()
                ]
                or [0]
            ),
            str(name),
        ),
        reverse=True,
    )[0]
    gene_sets = {
        term: clean_gene_set(genes)
        for term, genes in gp.get_library(
            name=reactome_library, organism=PATHWAY_ORGANISM
        ).items()
    }
    pathway_source = {
        "source": "reactome",
        "library": reactome_library,
        "organism": PATHWAY_ORGANISM,
        "gene_set_count": len(gene_sets),
    }

In [50]:
pathway_source

{'source': 'reactome',
 'library': 'Reactome_Pathways_2024',
 'organism': 'Mouse',
 'gene_set_count': 2100}

In [51]:
universe = de_table["gene"].astype(str).tolist()
up_genes = (
    de_table.loc[de_table["is_de"] & (de_table["log2FoldChange"] > 0), "gene"]
    .astype(str)
    .tolist()
)
down_genes = (
    de_table.loc[de_table["is_de"] & (de_table["log2FoldChange"] < 0), "gene"]
    .astype(str)
    .tolist()
)
ora_up = ora(up_genes, gene_sets, universe)
ora_down = ora(down_genes, gene_sets, universe)

In [53]:
fallback_score = np.sign(de_table["log2FoldChange"]) * -np.log10(
    np.clip(de_table["pvalue"], np.nextafter(0, 1), 1)
)
ranked = pd.DataFrame(
    {
        "gene": de_table["gene"].astype(str),
        "score": de_table["stat"].where(np.isfinite(de_table["stat"]), fallback_score),
    }
)
ranked = (
    ranked[np.isfinite(ranked["score"])]
    .drop_duplicates("gene")
    .sort_values("score", ascending=False)
)

In [56]:
gsea = gp.prerank(
    rnk=ranked,
    gene_sets=gene_sets,
    min_size=PATHWAY_MIN_OVERLAP,
    max_size=PATHWAY_MAX_SIZE,
    permutation_num=PATHWAY_GSEA_PERMUTATIONS,
    threads=N_CPUS,
    seed=PATHWAY_GSEA_SEED,
    outdir=None,
    no_plot=True,
    verbose=False,
).res2d.head(PATHWAY_TOP_N)

pathway_source, ora_up.head(), ora_down.head(), gsea.head()

({'source': 'reactome',
  'library': 'Reactome_Pathways_2024',
  'organism': 'Mouse',
  'gene_set_count': 2100},
                                        Term  Count  GeneRatio       P-value  \
 94                          Neuronal System     41   0.152985  3.311854e-14   
 8                             Axon Guidance     31   0.115672  1.216135e-05   
 91               Nervous System Development     31   0.115672  4.944222e-05   
 151   Transmission Across Chemical Synapses     25   0.093284  6.763826e-08   
 144  Signaling by Receptor Tyrosine Kinases     18   0.067164  4.969554e-01   
 
      Adjusted P-value  -log10 adjusted P-value  
 94       5.398323e-12                11.267741  
 8        2.202556e-04                 3.657073  
 91       5.372722e-04                 3.269806  
 151      5.512519e-06                 5.258650  
 144      7.864439e-01                 0.104332  ,
                                         Term  Count  GeneRatio       P-value  \
 59         Extracellul